# DX 704 Week 7 Project : Quadcopter and Linear Quadratic Regulator

This week's project will investigate issues in a quadcopter controller based using a <font color = 'cyan'>linear quadratic regulator.</font>
You will start with an existing model of the system and logs from a quadcopter based on it, investigate discrepancies, and ultimately train a new model of the system dynamics.


<font color = 'red'> FINAL SCORE: 80/80 </font>

The full project description and a template notebook are available on GitHub: [Project 7 Materials](https://github.com/bu-cds-dx704/dx704-project-07).


## Example Code

You may find it helpful to refer to these GitHub repositories of Jupyter notebooks for example code.

* https://github.com/bu-cds-omds/dx601-examples
* https://github.com/bu-cds-omds/dx602-examples
* https://github.com/bu-cds-omds/dx603-examples
* https://github.com/bu-cds-omds/dx704-examples

Any calculations demonstrated in code examples or videos may be found in these notebooks, and you are allowed to copy this example code in your homework answers.

## Imports

In [309]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.linalg import solve_discrete_are
from sklearn.linear_model import LinearRegression

# %pip install -q control
import control # type: ignore

## Functions

### simulate_solved

In [310]:
def simulate_solved(A, B, K, x_0, t_max):
    X   = [x_0]
    ts  = np.arange(0, t_max+1)
    for t in range(1, t_max+1):
        X.append(A @ X[t-1] + B @ -K @ X[t-1])

    X   = np.concatenate(X, axis=1)

    plt.plot(ts, X[0, :])
    plt.xlabel('Time (s)')
    plt.ylabel('Height')
    plt.show()

### simulate_q

In [311]:
def simulate_q(q):
    Q = np.array([[q, 0], [0, 0]])
    K, S, E = control.dlqr(A, B, Q, R)
    simulate_solved(A, B, K, x_0, t_max)

### simulate_r

In [312]:
def simulate_r(r):
    R = np.array([[r]])
    K, S, E = control.dlqr(A, B, Q, R)
    simulate_solved(A, B, K, x_0, t_max)

## Introduction

You've just joined a drone startup.
On your first day, you join your new team to watch a test flight for a new quadcopter prototype.
Watching the prototype fly, the team comments that it is not as smooth as usual and suspects that something is off in the controller.
They provide you a copy of the dynamics model and log data from the prototype to investigate.

The quadcopter control model is a slightly more sophisticated version of the 1D quadcopter problem previously considered.

The state vector $\mathbf{x}_t$ now includes an acceleration component, and the action vector now has a servo control for the throttle instead of a raw force component.
\begin{array}{rcl}
\mathbf{x}_t & = & \begin{bmatrix} x_t \\ v_t \\ a_t \end{bmatrix} \\ \\
\mathbf{u_t} & = & \begin{bmatrix} u_t \end{bmatrix}
\end{array}

In [313]:
model_A_df  = pd.read_csv('model-A.tsv', sep = '\t', index_col='index')
model_B_df  = pd.read_csv('model-B.tsv', sep = '\t', index_col='index')
cost_Q_df   = pd.read_csv('cost-Q.tsv', sep  = '\t', index_col='index')
cost_R_df   = pd.read_csv('cost-R.tsv', sep  = '\t', index_col='index')

In [314]:
model_A_df

,x,v,a
index,,,
x,1,1,0
v,0,1,1
a,0,0,1


In [315]:
model_B_df

,u
index,
x,0
v,0
a,1


In [316]:
cost_Q_df

,x,v,a
index,,,
x,5,0,0
v,0,1,0
a,0,0,2


In [317]:
cost_R_df

,u
index,
u,5


## Part 1: Reconstruct the Control Matrix

You are provided the dynamics model in the files `model-A.tsv`, `model-B.tsv`, `cost-Q.tsv` and `cost-R.tsv`.
Recompute the control matrix $\mathbf{K}$ to be used in the infinite horizon linear quadratic regulator problem.

In [318]:
# YOUR CHANGES HERE
 
# Convert DataFrames to NumPy arrays (matrices)
A = model_A_df.values
B = model_B_df.values
Q = cost_Q_df.values
R = cost_R_df.values

# Discrete-time linear quadratic regulator design 
K, S, E = control.dlqr(A, B, Q, R)
print("K = ", K, "\nk.shape = ", K.shape)

# control matrix K
K_df = pd.DataFrame(K, columns = ['x', 'v', 'a'])
K_df



K =  [[0.33445985 1.30445413 1.85813088]] 
k.shape =  (1, 3)


,x,v,a
0,0.33446,1.304454,1.858131


<font color = 'plum'>
This could also be done as follows:

1. Solve the Discrete Algebraic Riccati Equation (DARE) for P: 
`P = solve_discrete_are(A, B, Q, R)`

2. Calculate optimal feedback gain matrix K: 
`K = np.linalg.inv(B.T @ P @ B + R) @ B.T @ P @ A`


Save $\mathbf{K}$ in a file "`control-K-intended.tsv`" with columns `x`, `v` and `a`.

In [319]:
# YOUR CHANGES HERE

K_df.to_csv('control-K-intended.tsv', sep = '\t', index = False)

Submit "control-K-intended.tsv" in Gradescope.

## Part 2: Recompute the Actions for the Logged States

You get access to the log data for the prototype as the file "`qc-log.tsv`".
It conveniently saves all the state and actions made.
Recompute the actions based on your $\mathbf{K}$ matrix computed in part 1.

In [320]:
# YOUR CHANGES HERE

qc_log_df = pd.read_csv('qc-log.tsv', sep='\t')
qc_log_df

,index,x,v,a,u
0,0,-5.000000e+00,0.000000e+00,0.000000e+00,1.702188e+00
1,1,-5.000000e+00,-1.702188e-02,1.531969e+00,-1.263200e+00
2,2,-5.018724e+00,1.452683e+00,5.482855e-01,-1.249321e+00
3,3,-3.420773e+00,1.840779e+00,-5.212749e-01,-2.121274e-01
4,4,-1.395916e+00,1.163611e+00,-7.643170e-01,4.528954e-01
...,...,...,...,...,...
95,95,-1.855920e-18,1.097460e-18,-1.592936e-19,-4.843645e-19
96,96,-6.487133e-19,8.412290e-19,-6.111510e-19,3.189634e-19
97,97,2.766386e-19,1.733230e-19,-3.851990e-19,4.316642e-19
98,98,4.672940e-19,-2.142650e-19,-3.522116e-20,1.881712e-19


In [321]:
print(f'K = {K}, K.shape = {K.shape}')

# Extract the state vectors (x, v, a) from the log data
x_states = qc_log_df[['x', 'v', 'a']].values.T
print(f'x_states.shape = {x_states.shape}')
x_states

K = [[0.33445985 1.30445413 1.85813088]], K.shape = (1, 3)
x_states.shape = (3, 100)


array([[-5.00000000e+00, -5.00000000e+00, -5.01872407e+00,
        -3.42077294e+00, -1.39591608e+00, -1.15943786e-01,
         2.32338126e-01,  8.79599932e-02, -9.75906708e-02,
        -1.43030134e-01, -8.41271810e-02, -1.34536259e-02,
         1.92523381e-02,  1.70163142e-02,  3.33425558e-03,
        -5.17491703e-03, -5.53749997e-03, -2.16814042e-03,
         5.52753228e-04,  1.23513794e-03,  6.57081050e-04,
        -3.25997567e-05, -2.98543888e-04, -2.10461507e-04,
        -3.56629507e-05,  5.82073651e-05,  5.67893183e-05,
         1.78635782e-05, -1.00787303e-05, -1.47586247e-05,
        -6.81074403e-06,  9.62125440e-07,  3.46492289e-06,
         2.15405415e-06,  1.74670101e-07, -7.48384561e-07,
        -6.22781770e-07, -1.54794661e-07,  1.38904875e-07,
         1.65051869e-07,  6.50014630e-08, -1.92528834e-08,
        -4.06068018e-08, -2.20796197e-08,  2.42432819e-10,
         9.14903251e-09,  6.63582859e-09,  1.16663080e-09,
        -1.83818695e-09, -1.82757092e-09, -5.98544117e-1

In [322]:
# Recompute the actions u_check = -Kx
# u_check represents the actions we would have taken if the prototype had used our recomputed, intended control matrix
u_check = - (K @ x_states)
print(f'u_check.shape = {u_check.shape}')
print(f'u_check = {u_check}')


u_check.shape = (1, 100)
u_check = [[ 1.67229926e+00 -1.15209535e+00 -1.23518258e+00 -2.88503503e-01
   3.69201573e-01  4.30598487e-01  1.88643609e-01 -2.48031695e-02
  -8.90245580e-02 -5.23836416e-02 -7.81257443e-04  2.15504897e-02
   1.66200455e-02  3.70584348e-03 -3.90703556e-03 -4.32037493e-03
  -1.56055419e-03  6.23745949e-04  1.10220742e-03  5.62564833e-04
  -3.13639284e-05 -2.50460418e-04 -1.70992628e-04 -2.37612853e-05
   5.21739391e-05  4.81598406e-05  1.43010334e-05 -9.01791779e-06
  -1.24377051e-05 -5.48508923e-06  1.01996735e-06  2.98185907e-06
   1.78173856e-06  1.00441697e-07 -6.49869077e-07 -5.19119016e-07
  -1.18007362e-07  1.24124284e-07  1.39357909e-07  5.21638311e-08
  -1.82248395e-08 -3.46133286e-08 -1.80747701e-08  7.60467077e-10
   7.90565796e-09  5.51136963e-09  8.44844617e-10 -1.61772615e-09
  -1.53428984e-09 -4.72114029e-10  2.75953393e-10  3.95025546e-10
   1.78678549e-10 -2.89885863e-11 -9.39541590e-11 -5.73888381e-11
  -4.06796312e-12  2.03274485e-11  1.6618

In [323]:
# Add the recomputed actions to the DataFrame
qc_log_df['u_check'] = u_check.T
qc_log_df

,index,x,v,a,u,u_check
0,0,-5.000000e+00,0.000000e+00,0.000000e+00,1.702188e+00,1.672299e+00
1,1,-5.000000e+00,-1.702188e-02,1.531969e+00,-1.263200e+00,-1.152095e+00
2,2,-5.018724e+00,1.452683e+00,5.482855e-01,-1.249321e+00,-1.235183e+00
3,3,-3.420773e+00,1.840779e+00,-5.212749e-01,-2.121274e-01,-2.885035e-01
4,4,-1.395916e+00,1.163611e+00,-7.643170e-01,4.528954e-01,3.692016e-01
...,...,...,...,...,...,...
95,95,-1.855920e-18,1.097460e-18,-1.592936e-19,-4.843645e-19,-5.148677e-19
96,96,-6.487133e-19,8.412290e-19,-6.111510e-19,3.189634e-19,2.552224e-19
97,97,2.766386e-19,1.733230e-19,-3.851990e-19,4.316642e-19,3.971337e-19
98,98,4.672940e-19,-2.142650e-19,-3.522116e-20,1.881712e-19,1.886533e-19


Save your computed actions as "`qc-check.tsv`" with columns `index` and `u_check`.

In [324]:
# YOUR CHANGES HERE

qc_log_df[['index', 'u_check']].to_csv('qc-check.tsv', sep = '\t', index = False)

Submit "qc-check.tsv" in Gradescope.

## Part 3: Reverse Engineer the Actual Control Matrix

Now that you have found a systematic difference between your computed actions and the logged actions, estimate the control matrix that was actually used to choose actions in the prototype.

- Hint: With a linear quadratic regulator (LQR), the optimal actions are always linear combinations of the state that are calculated using the control matrix.

- You can use linear regression to reverse-engineer the coefficients in the control matrix.

<font color = 'plum'>Find the matrix `K` that relates the logged states to the logged actions. Use the `u` column from `qc-log.tsv` b/c it contains the actions that were actually executed by the prototype.

In [325]:
# YOUR CHANGES HERE

x_states
u_check
print(f'u_check.shape = {u_check.shape}, x_states.shape = {x_states.shape}')

u_check.shape = (1, 100), x_states.shape = (3, 100)


In [326]:
# # Separate states (X) and actions (y)
x_logged_states = qc_log_df[['x', 'v', 'a']].values
print(f'x_logged_states.shape: {x_logged_states.shape}')

u_logged_actions = qc_log_df['u'].values.reshape(-1, 1)
print(f'u_logged_actions.shape: {u_logged_actions.shape}')

# Fit the linear regression model to find the coefficients of u = Cx
model = LinearRegression()
model.fit(x_logged_states, u_logged_actions)

x_logged_states.shape: (100, 3)
u_logged_actions.shape: (100, 1)


,fit_intercept,True
,copy_X,True
,tol,1e-06
,n_jobs,None
,positive,False


In [327]:
# The coefficients from the model are C = -K_actual. So -C = K_actual
K_actual    = -1 * model.coef_
K_actual_df = pd.DataFrame(K_actual, columns=['x', 'v', 'a'])
K_actual_df

,x,v,a
0,0.340438,1.30012,1.950117


Save $\mathbf{K}_{\mathrm{actual}}$ in "`control-K-actual.tsv`" with the same format as "control-K-intended.tsv".

In [328]:
# YOUR CHANGES HERE

K_actual_df.to_csv('control-K-actual.tsv', sep = '\t', index = False)

Submit "actual-K.tsv" in Gradescope.

## Part 4: Recompute the System Dynamics from the Log Data

- On further investigation, it turns out that the control matrix $\mathbf{K}$ in the prototype was modified to compensate for state prediction errors.

- You would like to recompute the $\mathbf{A}$ and $\mathbf{B}$ matrices using the log data since they are used to predict the next states.

- However, since the action vector $\mathbf{u}_t$ is linearly dependent on the state via $\mathbf{u}_t=-\mathbf{K} \mathbf{x}_t$, you need a new data set so you can separate the effects of the $\mathbf{A}$ and $\mathbf{B}$ matrices.

- Your co-workers kindly provide a new file "`qc-train.tsv`" where noise was added to each action.

- Estimate the true $\mathbf{A}$ and $\mathbf{B}$ matrices based on this file.

In [329]:
# YOUR CHANGES HERE

# Use qc_train_df  to separate the effects of the A and B matrices. 
# Estimate the true A and B matrices based on this file.

qc_train_df = pd.read_csv('qc-train.tsv', sep='\t', index_col='index')
qc_train_df


,x,v,a,u
index,,,,
0,-5.000000,0.000000,0.000000,1.729856
1,-5.000000,-0.017299,1.556871,-1.311911
2,-5.019028,1.476577,0.531837,-1.198850
3,-3.394793,1.846154,-0.493944,-0.297565
4,-1.364024,1.195267,-0.811147,0.472619
...,...,...,...,...
95,0.089539,-0.162778,0.122030,-0.008091
96,-0.089517,-0.030491,0.126952,-0.174120
97,-0.123056,0.094904,-0.017061,0.038104


In [ ]:
#  feature matrix X includes current state and action
#  use all rows except the last one for input features
X_features = qc_train_df[['x', 'v', 'a', 'u']].iloc[:-1]
print(f'X_features.shape = {X_features.shape}')


#  target y includes the next state
# use all rows except the FIRST one for the target; 
# 'u' is an action (cause) and 'x' is the next state (effect), so 'u' has to be an input feature, not a target.
y_targets = qc_train_df[['x', 'v', 'a']].iloc[1:]
print(f'y_targets.shape = {y_targets.shape}')


# LinReg to solve for combined [A | B] matrix
model_dynamics = LinearRegression()
model_dynamics.fit(X_features, y_targets)

coeffs = model_dynamics.coef_
print(f'coeffs.shape = {coeffs.shape}')
print(f'coeffs = {coeffs}')

X_features.shape = (99, 4)
y_targets.shape = (99, 3)
coeffs.shape = (3, 4)
coeffs = [[ 1.00000000e+00  1.10000000e+00  1.58073643e-15 -4.43697230e-17]
 [ 1.65757515e-16  9.00000000e-01  9.50000000e-01 -1.00000000e-02]
 [ 4.49913254e-16  9.36033147e-16  1.10000000e+00  9.00000000e-01]]


In [331]:
# Extract the recomputed A and B matrices from the model coefficients
# The first three columns of the coefficients correspond to A

A_new = coeffs[:, :3]
print(f'A_new.shape = {A_new.shape}')
A_new

A_new.shape = (3, 3)


array([[1.00000000e+00, 1.10000000e+00, 1.58073643e-15],
       [1.65757515e-16, 9.00000000e-01, 9.50000000e-01],
       [4.49913254e-16, 9.36033147e-16, 1.10000000e+00]])

In [332]:
# The last column of the coefficients corresponds to B
B_new = coeffs[:, 3:]
print(f'B_new.shape = {B_new.shape}')
B_new

B_new.shape = (3, 1)


array([[-4.4369723e-17],
       [-1.0000000e-02],
       [ 9.0000000e-01]])

In [333]:
A_new_df = pd.DataFrame(A_new, columns=['x', 'v', 'a'], index=['x', 'v', 'a'])
B_new_df = pd.DataFrame(B_new, columns=['u'], index=['x', 'v', 'a'])

In [334]:
A_new_df

,x,v,a
x,1.000000e+00,1.100000e+00,1.580736e-15
v,1.657575e-16,9.000000e-01,9.500000e-01
a,4.499133e-16,9.360331e-16,1.100000e+00


In [335]:
B_new_df

,u
x,-4.436972e-17
v,-1.000000e-02
a,9.000000e-01


Save $\mathbf{A}$ and $\mathbf{B}$ to "`model-A-new.tsv`" and "`model-B-new.tsv`" respectively.

In [336]:
# YOUR CHANGES HERE

A_new_df.to_csv('model-A-new.tsv', sep = '\t', index = True)
B_new_df.to_csv('model-B-new.tsv', sep = '\t', index = True)

Submit "model-A-new.tsv" and "model-B-new.tsv" in Gradescope.

## Part 5: Code

Please submit a Jupyter notebook that can reproduce all your calculations and recreate the previously submitted files.
You do not need to provide code for data collection if you did that by manually.

## Part 6: Acknowledgements

If you discussed this assignment with anyone, please acknowledge them here.
If you did this assignment completely on your own, simply write none below.

If you used any libraries not mentioned in this module's content, please list them with a brief explanation what you used them for. If you did not use any other libraries, simply write none below.

If you used any generative AI tools, please add links to your transcripts below, and any other information that you feel is necessary to comply with the generative AI policy. If you did not use any generative AI tools, simply write none below.

## Notes

<font color = 'plum'> 

### *What's a linear quadratic regulator and when would we use it ?* 
https://g.co/gemini/share/00bd802ee074

A **Linear-Quadratic Regulator (LQR)** is a type of **optimal control** method used to find the best way to operate a dynamic system at the lowest possible "cost." It's one of the most fundamental and widely used results in optimal control theory.

#### How LQR Works 

The core of LQR is an **optimization problem**. It works by defining a **quadratic cost function** that represents the desired performance. This function is typically made up of two main parts:

1.  **State Deviation Cost**: A penalty for how far the system's states (e.g., position, velocity) are from their desired values.
2.  **Control Effort Cost**: A penalty for the amount of control input (e.g., motor power, steering angle) required.

The LQR algorithm then calculates a constant gain matrix, **K**, which is used in a feedback control law to minimize this cost function. The control input, **u**, is a linear function of the system's current state, **x**, as expressed by the equation: $u = -Kx$. The value of **K** is found by solving a mathematical problem known as the **Algebraic Riccati Equation**.

A key advantage of LQR is that it provides a systematic way to design a controller that balances these competing objectives: achieving a desired state quickly versus using minimal energy or control effort. The engineer can adjust the relative importance of these costs by tuning the weighting matrices, **Q** and **R**, in the cost function.

***

#### When LQR is Used 

LQR is a powerful tool for controlling systems that can be described by **linear differential equations**. It's particularly useful for stabilizing systems around an equilibrium point. Even for **nonlinear systems**, LQR can be applied by linearizing the system's dynamics around a specific operating point or trajectory.

Some common real-world applications include:

* **Aerospace**: Stabilizing aircraft flight, controlling satellites, or optimizing rocket trajectories.
* **Robotics**: Balancing and stabilizing robotic systems like an inverted pendulum on a cart, or enabling wheeled robots and drones to follow a path or hover at a specific altitude.
* **Automotive**: Implementing cruise control or lane-keeping assistance in self-driving cars.
* **Process Control**: Maintaining desired temperatures, pressures, or fluid levels in chemical plants or manufacturing processes.

While LQR is a powerful technique, it assumes the system is linear, time-invariant, and that all system states are fully measurable. In practice, LQR is often combined with other techniques, like a **Kalman filter**, to estimate states that aren't directly measurable. This combined approach is known as **LQG (Linear-Quadratic-Gaussian)** control.

---


<font color = 'plum'>
An optimal control problem seeks to find the best control inputs, or actions, to minimize a given cost function while adhering to a system's dynamics. The infinite horizon Linear Quadratic Regulator (LQR) is a method for solving this problem when the system dynamics are linear and the cost function is quadratic. The LQR finds a time-invariant feedback gain matrix

$\mathbf{K}$ that maps the current state to the optimal action, such that  $\mathbf{u} = -\mathbf{K}\mathbf{x}$.

The discrete-time linear dynamics are described by the equations:
$\mathbf{x}_{t+1} = \mathbf{A}\mathbf{x}_t + \mathbf{B}\mathbf{u}_t$


The cost function to minimize is:
$J = \sum_{t=0}^{\infty} (\mathbf{x}_t^\top\mathbf{Q}\mathbf{x}_t + \mathbf{u}_t^\top\mathbf{R}\mathbf{u}_t)$

In the provided files, the matrices are:
* $\mathbf{A}$ is `model-A.tsv` 
* $\mathbf{B}$ is `model-B.tsv` 
* $\mathbf{Q}$ is `cost-Q.tsv` 
* $\mathbf{R}$ is `cost-R.tsv` 
* The log data is in `qc-log.tsv` 

The dimensions of the matrices are:
* State vector $\mathbf{x} \in \mathbb{R}^3$ (x, v, a)
* Action vector $\mathbf{u} \in \mathbb{R}^1$ (u)
* State transition matrix $\mathbf{A} \in \mathbb{R}^{3 \times 3}$
* Control matrix $\mathbf{B} \in \mathbb{R}^{3 \times 1}$
* State cost matrix $\mathbf{Q} \in \mathbb{R}^{3 \times 3}$
* Action cost matrix $\mathbf{R} \in \mathbb{R}^{1 \times 1}$
* Feedback gain matrix $\mathbf{K} \in \mathbb{R}^{1 \times 3}$

***

